# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [1]:
!pip install --upgrade --quiet datasets fsspec huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.0/521.0 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.2.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.


In [2]:
!pip uninstall -y huggingface-hub transformers datasets
!pip install huggingface-hub==0.35.1 transformers==4.45.2 datasets==2.21.0


Found existing installation: huggingface_hub 1.2.3
Uninstalling huggingface_hub-1.2.3:
  Successfully uninstalled huggingface_hub-1.2.3
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: datasets 4.4.2
Uninstalling datasets-4.4.2:
  Successfully uninstalled datasets-4.4.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.3/563.3 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 107.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully unins

In [ ]:
import os
os.kill(os.getpid(), 9)


---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [7]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm


print("✅ Import 성공")


✅ Import 성공


In [2]:
# batch_size와 epochs를 조정해보세요!
batch_size = 16
epochs = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [3]:
# 데이터셋 로드
raw_datasets = load_dataset("sst2")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [4]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [5]:
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    # tokenizer / model 로드
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    ).to(device)

    # 데이터 전처리
    tokenized_datasets = raw_datasets.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True
    )

    tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])
    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch")

    train_dataset = tokenized_datasets["train"]
    valid_dataset = tokenized_datasets["validation"]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    # ===== 학습 =====
    model.train()
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        for batch in tqdm(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ===== 평가 =====
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)

            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")

    return acc



In [8]:
# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")

print("\n===== Final Results =====")
print(f"BERT Accuracy     : {bert_acc:.4f}")
print(f"ELECTRA Accuracy  : {electra_acc:.4f}")


======== Now Training: bert-base-uncased ========


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/872 [00:00<?, ? examples/s]


Epoch 1/2


100%|██████████| 4210/4210 [23:02<00:00,  3.04it/s]


...


Epoch 1/2


100%|██████████| 4210/4210 [23:16<00:00,  3.01it/s]



Epoch 2/2


100%|██████████| 4210/4210 [23:16<00:00,  3.01it/s]


Validation Accuracy (google/electra-base-discriminator): 0.9392

===== Final Results =====
BERT Accuracy     : 0.9163
ELECTRA Accuracy  : 0.9392


## 📊 결과 비교 및 분석

아래 항목에 대한 답을 간략히 적어주세요:

1. 각 모델 구조 설명

BERT (Bidirectional Encoder Representations from Transformers)
BERT는 Transformer encoder 구조를 기반으로 한 사전학습 언어모델로, 문장 내 단어를 양방향 문맥을 고려하여 표현한다. 사전학습 단계에서 Masked Language Modeling과 Next Sentence Prediction을 통해 언어 표현을 학습하며, 이후 분류 문제에서는 출력 벡터 위에 분류기 층을 추가하여 fine-tuning한다.

ELECTRA (Efficiently Learning an Encoder that Classifies Token Replacements Accurately)
ELECTRA는 Generator–Discriminator 구조를 사용한다. Generator는 일부 토큰을 대체하고, Discriminator는 각 토큰이 원래 토큰인지 대체된 토큰인지를 판별하도록 학습된다. 이 방식은 모든 토큰에 대해 학습 신호를 제공하므로, BERT보다 샘플 효율적인 사전학습이 가능하다.

2. 어떤 모델이 적합한지에 대한 본인의 의견
  - 학습 속도, accuracy 등 고려

본 실험에서는 ELECTRA가 BERT보다 더 높은 정확도를 보였으며, 동일한 데이터셋에서 더 효율적으로 학습되었다. 따라서 계산 자원이 제한된 환경이나 비교적 작은 데이터셋에서도 높은 성능을 기대할 수 있는 ELECTRA가 감성 분류 과제에 더 적합하다고 판단된다.

BERT는 구조가 직관적이고 안정적인 성능을 제공하지만, Masked Language Modeling 특성상 학습 효율이 상대적으로 낮다. 반면 ELECTRA는 모든 토큰을 학습에 활용하므로 학습 속도가 빠르고, 동일한 epoch 수에서도 더 높은 accuracy를 달성하였다. 본 실험 결과(BERT: 0.9163, ELECTRA: 0.9392)를 고려할 때, 성능과 효율 측면 모두에서 ELECTRA가 우수하였다.

-
-